# TP0 — Mon tout premier neurone artificiel
**Cours IA & Cybersécurité — Master 1**

---

## Ce que vous allez faire

Dans ce TP, vous allez construire **un seul neurone** capable de distinguer du trafic réseau **normal** du trafic **suspect**.

Pas de bibliothèque magique. Juste Python, 4 exemples, et des maths de collège.

**Durée** : 1h  
**Prérequis** : savoir ce qu'est une variable, une boucle `for`, une fonction

---

## Le problème

Un analyste SOC observe 4 connexions réseau. Il les a classées à la main :

| Connexion | Paquets/sec | Taille moy. (octets) | Verdict |
|-----------|-------------|----------------------|---------|
| A         | 1           | 1                    | Normal  |
| B         | 1           | 2                    | Normal  |
| C         | 2           | 1                    | Normal  |
| D         | 2           | 2                    | **Suspect** |

Votre neurone doit **apprendre** cette règle tout seul, sans qu'on lui explique pourquoi D est suspect.

---
## Étape 1 — Représenter les données

En machine learning, chaque exemple est un **vecteur** de chiffres.  
Ici chaque connexion a 2 caractéristiques (features) : paquets/sec et taille.

On note :
- `X` = la liste de tous les exemples
- `y` = la liste des réponses correctes (0 = normal, 1 = suspect)

In [ ]:
# Les 4 exemples : [paquets_par_sec, taille_moyenne]
X = [
    [1, 1],   # Connexion A — normale
    [1, 2],   # Connexion B — normale
    [2, 1],   # Connexion C — normale
    [2, 2],   # Connexion D — suspecte
]

# Les réponses correctes (0 = normal, 1 = suspect)
y = [0, 0, 0, 1]

print("Données chargées !")
print(f"Nombre d'exemples : {len(X)}")
print(f"Exemple A : features={X[0]}, label={y[0]}")
print(f"Exemple D : features={X[3]}, label={y[3]}")

---
## Étape 2 — Comprendre le neurone

Un neurone artificiel fait **3 choses** :

```
  x1 (paquets/sec)  ──× w1──┐
                             ├──▶  z = x1×w1 + x2×w2 + b  ──▶  activation  ──▶  0 ou 1
  x2 (taille)       ──× w2──┘
                    + biais b
```

1. **Pondérer** chaque entrée par un poids (`w1`, `w2`)
2. **Sommer** le tout + un biais `b` → on appelle ça `z`
3. **Décider** : si `z > 0` → suspect (1), sinon → normal (0)

Au départ, `w1`, `w2` et `b` sont aléatoires. Le neurone va les **ajuster** à chaque erreur.

In [ ]:
import random
random.seed(42)

# Initialisation aléatoire des poids (petites valeurs)
w1 = random.uniform(-0.5, 0.5)
w2 = random.uniform(-0.5, 0.5)
b  = 0.0  # biais initialisé à zéro

print("Poids initiaux (aléatoires) :")
print(f"  w1 = {w1:.4f}")
print(f"  w2 = {w2:.4f}")
print(f"  b  = {b:.4f}")

---
## Étape 3 — La fonction de décision

On calcule `z = x1×w1 + x2×w2 + b`, puis on applique un **seuil** :
- Si `z > 0` → le neurone dit **suspect** (1)
- Sinon → il dit **normal** (0)

C'est la **fonction d'activation échelon** (step function).

### À vous de jouer !

Complétez les deux fonctions ci-dessous.

In [ ]:
def calculer_z(x1, x2, w1, w2, b):
    """
    Calcule la somme pondérée z = x1*w1 + x2*w2 + b
    """
    z = x1*w1 + x2*w2 + b  # ← complétez cette ligne
    return z


def activer(z):
    """
    Fonction d'activation échelon :
    - retourne 1 si z > 0
    - retourne 0 sinon
    """
    if z>0:       # ← complétez la condition
        return 1
    else:
        return 0


# Test rapide
z_test = calculer_z(2, 2, w1, w2, b)
pred_test = activer(z_test)
print(f"Test sur connexion D [2,2] : z={z_test:.4f}, prédiction={pred_test}")

---
## Étape 4 — Tester le neurone AVANT apprentissage

Regardons ce que prédit le neurone avec ses poids aléatoires.

In [ ]:
noms = ['A', 'B', 'C', 'D']

print("Prédictions AVANT apprentissage :")
print(f"{'Connexion':10s} {'Réel':8s} {'Prédit':8s} {'Résultat'}")
print("-" * 40)

for i in range(len(X)):
    x1, x2 = X[i][0], X[i][1]
    z = calculer_z(x1, x2, w1, w2, b)
    prediction = activer(z)
    correct = "OK" if prediction == y[i] else "ERREUR"
    print(f"{noms[i]:10s} {y[i]:8d} {prediction:8d} {correct}")

---
## Étape 5 — La règle d'apprentissage

Quand le neurone se trompe, on corrige les poids selon cette règle simple :

```
erreur = réponse_correcte - prédiction

w1 = w1 + lr × erreur × x1
w2 = w2 + lr × erreur × x2
b  = b  + lr × erreur
```

- Si `erreur = 0` → le neurone avait juste, on ne change rien
- Si `erreur = +1` → il a dit 0 alors que c'était 1 → on **augmente** les poids
- Si `erreur = -1` → il a dit 1 alors que c'était 0 → on **diminue** les poids

`lr` est le **taux d'apprentissage** (learning rate) : il contrôle la vitesse de correction.

### À vous de jouer !

Complétez la fonction de mise à jour des poids.

In [ ]:
def mettre_a_jour(w1, w2, b, x1, x2, erreur, lr):
    """
    Met à jour les poids selon la règle du perceptron.
    
    Paramètres :
        w1, w2 : poids actuels
        b      : biais actuel
        x1, x2 : features de l'exemple courant
        erreur : réponse_correcte - prédiction
        lr     : taux d'apprentissage
    
    Retourne :
        w1, w2, b mis à jour
    """
    nouveau_w1 = w1 + lr*erreur*x1   # ← appliquez la formule
    nouveau_w2 = w2 + lr*erreur*x2   # ← appliquez la formule
    nouveau_b  = b  + lr*erreur   # ← appliquez la formule
    
    return nouveau_w1, nouveau_w2, nouveau_b


# Test : on simule une erreur = +1 sur l'exemple D [2, 2]
w1_test, w2_test, b_test = mettre_a_jour(w1, w2, b, 2, 2, erreur=1, lr=0.1)
print("Après une correction sur D :")
print(f"  w1 : {w1:.4f} → {w1_test:.4f}")
print(f"  w2 : {w2:.4f} → {w2_test:.4f}")
print(f"  b  : {b:.4f} → {b_test:.4f}")

---
## Étape 6 — La boucle d'apprentissage

On répète plusieurs fois (**époques**) le cycle suivant :

```
Pour chaque exemple :
    1. Calculer z
    2. Prédire (activer)
    3. Calculer l'erreur
    4. Mettre à jour les poids si erreur ≠ 0
```

### À vous de jouer !

Complétez la boucle d'apprentissage.

In [ ]:
# Hyperparamètres
lr     = 0.1   # taux d'apprentissage
epochs = 20    # nombre de passes sur le dataset

# On repart des poids initiaux
w1 = random.uniform(-0.5, 0.5)
w2 = random.uniform(-0.5, 0.5)
b  = 0.0

historique_erreurs = []  # pour tracer la courbe d'apprentissage

print(f"{'Époque':8s} {'Erreurs':10s} {'w1':10s} {'w2':10s} {'b':10s}")
print("-" * 50)

for epoch in range(epochs):
    nb_erreurs = 0
    
    for i in range(len(X)):
        x1, x2 = X[i][0], X[i][1]
        
        # 1. Calculer z
        z = calculer_z(x1, x2, nouveau_w1, nouveau_w2, nouveau_b)           # ← complétez les arguments
        
        # 2. Prédire
        prediction = activer(z)     # ← complétez
        
        # 3. Calculer l'erreur
        erreur = réponse_correcte - prédiction            # ← réponse_correcte - prédiction
        
        # 4. Mettre à jour si erreur
        if erreur != 0:
            w1, w2, b = mettre_a_jour(z)   # ← complétez les arguments
            nb_erreurs += 1
    
    historique_erreurs.append(nb_erreurs)
    print(f"{epoch+1:8d} {nb_erreurs:10d} {w1:10.4f} {w2:10.4f} {b:10.4f}")

---
## Étape 7 — Résultats APRÈS apprentissage

In [ ]:
print("Prédictions APRÈS apprentissage :")
print(f"{'Connexion':10s} {'Réel':8s} {'Prédit':8s} {'Résultat'}")
print("-" * 40)

nb_corrects = 0
for i in range(len(X)):
    x1, x2 = X[i][0], X[i][1]
    z = calculer_z(x1, x2, w1, w2, b)
    prediction = activer(z)
    correct = "OK" if prediction == y[i] else "ERREUR"
    if prediction == y[i]:
        nb_corrects += 1
    print(f"{noms[i]:10s} {y[i]:8d} {prediction:8d} {correct}")

print(f"\nAccuracy : {nb_corrects}/{len(X)} = {nb_corrects/len(X):.0%}")
print(f"Poids finaux : w1={w1:.4f}, w2={w2:.4f}, b={b:.4f}")

---
## Étape 8 — Visualisation de l'apprentissage

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Graphique 1 : courbe d'apprentissage
ax1 = axes[0]
ax1.plot(range(1, epochs+1), historique_erreurs,
         marker='o', color='slateblue', linewidth=2, markersize=6)
ax1.fill_between(range(1, epochs+1), historique_erreurs, alpha=0.15, color='slateblue')
ax1.set_xlabel('Époque')
ax1.set_ylabel('Nombre d\'erreurs')
ax1.set_title('Courbe d\'apprentissage')
ax1.set_ylim(-0.1, max(historique_erreurs) + 0.5)
ax1.grid(alpha=0.3)

# ── Graphique 2 : frontière de décision
ax2 = axes[1]

# Points de données
for i in range(len(X)):
    couleur = 'tomato' if y[i] == 1 else 'steelblue'
    label   = 'Suspect' if y[i] == 1 else 'Normal'
    ax2.scatter(X[i][0], X[i][1], c=couleur, s=200, zorder=5,
                edgecolors='white', linewidth=1.5)
    ax2.annotate(noms[i], (X[i][0], X[i][1]),
                 textcoords='offset points', xytext=(8, 5), fontsize=11)

# Frontière de décision : z = w1*x1 + w2*x2 + b = 0
# => x2 = (-w1*x1 - b) / w2
import numpy as np
x1_vals = np.linspace(0.5, 2.5, 100)
if abs(w2) > 1e-6:
    x2_vals = (-w1 * x1_vals - b) / w2
    ax2.plot(x1_vals, x2_vals, 'k--', linewidth=1.5, label='Frontière de décision')

# Légende manuelle
from matplotlib.lines import Line2D
legende = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue', markersize=10, label='Normal'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='tomato',    markersize=10, label='Suspect'),
    Line2D([0],[0], linestyle='--', color='black', label='Frontière'),
]
ax2.legend(handles=legende, loc='upper left')
ax2.set_xlabel('Paquets / sec')
ax2.set_ylabel('Taille moyenne (octets)')
ax2.set_title('Frontière de décision apprise')
ax2.set_xlim(0.5, 2.5)
ax2.set_ylim(0.5, 2.5)
ax2.grid(alpha=0.3)

plt.suptitle('Résultats du perceptron', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Étape 9 — Questions de réflexion

Répondez dans la cellule Markdown ci-dessous.

**Question 1** — Que se passe-t-il si `erreur = 0` ? Pourquoi ne met-on pas à jour les poids ?

*Votre réponse :* ...

---

**Question 2** — Dans la courbe d'apprentissage, à quelle époque le neurone fait-il 0 erreur ?

*Votre réponse :* ...

---

**Question 3** — Essayez `lr = 1.0` puis `lr = 0.001`. Que se passe-t-il ? Pourquoi ?

*Votre réponse :* ...

---
## Étape 10 — Bonus : tester une nouvelle connexion

Le neurone a appris. Maintenant, analysez une connexion inconnue.

In [ ]:
# Nouvelle connexion à analyser : 3 paquets/sec, taille 3
nouvelle_connexion = [3, 3]

x1_new, x2_new = nouvelle_connexion
z_new   = calculer_z(x1_new, x2_new, w1, w2, b)
pred_new = activer(z_new)

verdict = "SUSPECT" if pred_new == 1 else "NORMAL"
print(f"Connexion {nouvelle_connexion} → z={z_new:.4f} → {verdict}")

# À vous : testez d'autres valeurs !
# nouvelle_connexion = [1, 1]  ← normal ?
# nouvelle_connexion = [3, 1]  ← suspect ?

---
## Récapitulatif — Ce que vous avez fait

| Étape | Ce que vous avez appris |
|-------|-------------------------|
| 1 | Représenter des données sous forme de vecteurs |
| 2-3 | Calculer la somme pondérée `z` et appliquer un seuil |
| 4 | Tester un neurone non entraîné (résultats aléatoires) |
| 5 | Comprendre la règle de mise à jour des poids |
| 6 | Écrire la boucle d'apprentissage complète |
| 7-8 | Évaluer et visualiser les résultats |

**Prochaine étape → TP1** : même principe mais sur 200 exemples, avec NumPy et PyTorch.

---

**Rendu** : sauvegardez (`Ctrl+S`) et déposez `tp0_NOM_Prenom.ipynb` sur le portail.